# CMIP6-decadal monthly `tas` concat/subset comparison

When: 2026-08-21

This notebook runs the same request against the DKRZ and IPSL WPS services, which deploy different Rook versions. It processes ten monthly CMIP6-decadal near-surface air temperature (`tas`) realizations from HadGEM3-GC31-MM, concatenates them along `realization`, and subsets the two complete calendar years 1996–1997.

Both results are downloaded and compared in the final section.


## Configuration

Keep the endpoint mapping fixed while comparing versions. `Operator.orchestrate()` reinitializes the Rooki client from `ROOK_URL` for every request, so the same workflow can be submitted to both services in one kernel.


In [1]:
import json
import os
import numpy as np
import xarray as xr

WPS_SERVICES = {
    "DKRZ": "http://rook.dkrz.de/wps",
    "IPSL": "http://copernicus-wps.ipsl.fr/wps",
}

# Importing operators initializes a client, so select a known endpoint first.
os.environ["ROOK_URL"] = WPS_SERVICES["DKRZ"]
from rooki import operators as ops


In [2]:
TIME_RANGE = "1996/1997"

collection = [
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r1i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r2i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r3i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r4i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r5i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r6i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r7i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r8i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r9i1p1f2.Amon.tas.gn.v20200417",
    "c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r10i1p1f2.Amon.tas.gn.v20200417",
]


## Concatenate and subset the realizations

Build and inspect a workflow that concatenates along `realization` and then selects the two complete calendar years 1996 and 1997.


In [3]:
tas = ops.Input("tas", collection)
concat = ops.Concat(tas, dims="realization", time=TIME_RANGE)
workflow = ops.Subset(concat, time=TIME_RANGE)

serialized_request = json.loads(workflow._serialise())
serialized_request


{'inputs': {'tas': ['c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r1i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r2i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r3i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r4i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r5i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r6i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r7i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r8i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1995-r9i1p1f2.Amon.tas.gn.v20200417',
   'c3s-cmip6-decadal.DCPP.MOHC.HadGEM3-GC31-MM.dcppA-hindcast.s1

## Run the request on both services

Submit the identical serialized workflow to each endpoint and show only its download URL or URLs.


In [4]:
responses = {}

for service, url in WPS_SERVICES.items():
    os.environ["ROOK_URL"] = url
    response = workflow.orchestrate()
    responses[service] = response

    print(f"{service}:")
    if response.ok:
        for download_url in response.download_urls():
            print(download_url)


DKRZ:
http://rook7.cloud.dkrz.de:80/outputs/rook/c1ebaf30-9d96-11f1-b77c-fa163eb671ca/tas_Amon_HadGEM3-GC31-MM_dcppA-hindcast_r10i1p1f2_gn_19960116-19961216.nc
http://rook7.cloud.dkrz.de:80/outputs/rook/c1ebc114-9d96-11f1-b77c-fa163eb671ca/tas_Amon_HadGEM3-GC31-MM_dcppA-hindcast_r10i1p1f2_gn_19970116-19971216.nc
IPSL:
http://copernicus-wps.ipsl.fr:80/outputs/rook/caf89822-9d96-11f1-80f5-0050568fd52c/tas_Amon_HadGEM3-GC31-MM_dcppA-hindcast_r10i1p1f2_gn_19960116-19971216.nc


## Download and combine each service's results

DKRZ returns one batched file per requested year, while IPSL returns both years in one file. Download every output and use `xarray.open_mfdataset(..., combine="by_coords")` for both services so each result is represented by one logical dataset before comparison.


In [5]:
result_files = {}
datasets = {}

for service, response in responses.items():
    if not response.ok:
        continue
    files = response.download()
    if not files:
        print(f"{service}: no result files were downloaded")
        continue
    result_files[service] = files
    datasets[service] = xr.open_mfdataset(
        files,
        combine="by_coords",
        data_vars="all",
        decode_timedelta=True,
    )


## Compare structure, time, values, and decadal metadata

The checks below report differences explicitly instead of reducing the result to one boolean. In particular, they inspect the decadal coordinates `realization`, `reftime`, and `leadtime`, following the earlier decadal notebooks.


In [6]:
required_services = set(WPS_SERVICES)
if required_services <= datasets.keys():
    dkrz = datasets["DKRZ"]
    ipsl = datasets["IPSL"]
    print("Dataset comparison (DKRZ vs IPSL)")
    print(f"  identical, including attributes: {dkrz.identical(ipsl)}")
    print(f"  equal, ignoring attributes:       {dkrz.equals(ipsl)}")
    print(f"  same dimensions:                  {dict(dkrz.sizes) == dict(ipsl.sizes)}")
    print(f"  same coordinate names:            {set(dkrz.coords) == set(ipsl.coords)}")
else:
    missing = required_services - datasets.keys()
    print(f"Cannot compare datasets; missing successful result(s): {sorted(missing)}")


Dataset comparison (DKRZ vs IPSL)
  identical, including attributes: False
  equal, ignoring attributes:       False
  same dimensions:                  True
  same coordinate names:            True


In [7]:
COORDINATES_TO_COMPARE = ["time", "realization", "reftime", "leadtime"]

if required_services <= datasets.keys():
    for coordinate_name in COORDINATES_TO_COMPARE:
        print(f"\n{coordinate_name}")
        if coordinate_name not in dkrz.coords or coordinate_name not in ipsl.coords:
            print(f"  present in DKRZ: {coordinate_name in dkrz.coords}")
            print(f"  present in IPSL: {coordinate_name in ipsl.coords}")
            continue

        dkrz_coordinate = dkrz[coordinate_name]
        ipsl_coordinate = ipsl[coordinate_name]
        print(f"  same dimensions: {dkrz_coordinate.dims == ipsl_coordinate.dims}")
        print(f"  same dtype:      {dkrz_coordinate.dtype == ipsl_coordinate.dtype}")
        print(
            "  same values:     "
            f"{np.array_equal(dkrz_coordinate.values, ipsl_coordinate.values)}"
        )
        print(f"  same attributes: {dkrz_coordinate.attrs == ipsl_coordinate.attrs}")
        print(f"  DKRZ values: {dkrz_coordinate.values!r}")
        print(f"  IPSL values: {ipsl_coordinate.values!r}")
        print(f"  DKRZ attributes: {dkrz_coordinate.attrs!r}")
        print(f"  IPSL attributes: {ipsl_coordinate.attrs!r}")
        if coordinate_name == "time":
            encoding_keys = ("calendar", "units", "dtype")
            dkrz_encoding = {
                key: dkrz_coordinate.encoding.get(key) for key in encoding_keys
            }
            ipsl_encoding = {
                key: ipsl_coordinate.encoding.get(key) for key in encoding_keys
            }
            print(f"  DKRZ time encoding: {dkrz_encoding!r}")
            print(f"  IPSL time encoding: {ipsl_encoding!r}")



time
  same dimensions: True
  same dtype:      True
  same values:     True
  same attributes: True
  DKRZ values: array([cftime.Datetime360Day(1996, 1, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 2, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 3, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 4, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 5, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 6, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 7, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 8, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 9, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 10, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 11, 16, 0, 0, 0, 0, has_year_zero=True),
       cftime.Datetime360Day(1996, 12, 16, 0, 0, 0, 0, h

### Compare attributes

First show the DCPP/decadal global attributes side by side, including missing values. Then report every difference in the complete global, `tas`, and decadal-coordinate attribute mappings.


In [8]:
DECADAL_ATTRIBUTES = [
    "activity_id",
    "experiment",
    "experiment_id",
    "sub_experiment",
    "sub_experiment_id",
    "initialization_index",
    "realization_index",
    "forcing_index",
    "physics_index",
    "variant_label",
    "startdate",
    "parent_activity_id",
    "parent_experiment_id",
    "parent_mip_era",
    "parent_source_id",
    "parent_time_units",
    "branch_method",
    "branch_time_in_child",
    "branch_time_in_parent",
]
MISSING = "<missing>"

def attribute_values_equal(left, right):
    try:
        return bool(np.all(left == right))
    except Exception:
        return False

def report_attribute_differences(label, dkrz_attributes, ipsl_attributes):
    differences = []
    for name in sorted(set(dkrz_attributes) | set(ipsl_attributes)):
        dkrz_value = dkrz_attributes.get(name, MISSING)
        ipsl_value = ipsl_attributes.get(name, MISSING)
        if not attribute_values_equal(dkrz_value, ipsl_value):
            differences.append((name, dkrz_value, ipsl_value))

    print(f"\n{label}: {len(differences)} attribute difference(s)")
    for name, dkrz_value, ipsl_value in differences:
        print(f"  {name}")
        print(f"    DKRZ: {dkrz_value!r}")
        print(f"    IPSL: {ipsl_value!r}")

def report_all_attributes(label, dkrz_attributes, ipsl_attributes):
    print(f"\n{label}")
    names = sorted(set(dkrz_attributes) | set(ipsl_attributes))
    if not names:
        print("  <no attributes>")
        return
    for name in names:
        dkrz_value = dkrz_attributes.get(name, MISSING)
        ipsl_value = ipsl_attributes.get(name, MISSING)
        same = attribute_values_equal(dkrz_value, ipsl_value)
        print(f"  {name}: same={same}")
        print(f"    DKRZ: {dkrz_value!r}")
        print(f"    IPSL: {ipsl_value!r}")

if required_services <= datasets.keys():
    print("DCPP/decadal global attributes")
    for name in DECADAL_ATTRIBUTES:
        dkrz_value = dkrz.attrs.get(name, MISSING)
        ipsl_value = ipsl.attrs.get(name, MISSING)
        same = attribute_values_equal(dkrz_value, ipsl_value)
        print(f"  {name}: same={same}")
        print(f"    DKRZ: {dkrz_value!r}")
        print(f"    IPSL: {ipsl_value!r}")

    report_attribute_differences("Global attributes", dkrz.attrs, ipsl.attrs)
    report_attribute_differences("tas attributes", dkrz["tas"].attrs, ipsl["tas"].attrs)
    for coordinate_name in ("time", "realization", "reftime", "leadtime"):
        if coordinate_name in dkrz.coords and coordinate_name in ipsl.coords:
            report_attribute_differences(
                f"{coordinate_name} attributes",
                dkrz[coordinate_name].attrs,
                ipsl[coordinate_name].attrs,
            )

    print("\nComplete attribute comparison")
    report_all_attributes("Global attributes", dkrz.attrs, ipsl.attrs)
    for variable_name in sorted(set(dkrz.variables) | set(ipsl.variables)):
        if variable_name not in dkrz.variables or variable_name not in ipsl.variables:
            print(f"\n{variable_name}: present in DKRZ={variable_name in dkrz.variables}, present in IPSL={variable_name in ipsl.variables}")
            continue
        report_all_attributes(
            f"{variable_name} attributes",
            dkrz[variable_name].attrs,
            ipsl[variable_name].attrs,
        )


DCPP/decadal global attributes
  activity_id: same=True
    DKRZ: 'DCPP'
    IPSL: 'DCPP'
  experiment: same=True
    DKRZ: 'hindcast initialized based on observations and using historical forcing'
    IPSL: 'hindcast initialized based on observations and using historical forcing'
  experiment_id: same=True
    DKRZ: 'dcppA-hindcast'
    IPSL: 'dcppA-hindcast'
  sub_experiment: same=True
    DKRZ: 'initialized near end of year 1995'
    IPSL: 'initialized near end of year 1995'
  sub_experiment_id: same=True
    DKRZ: 's199511'
    IPSL: 's199511'
  initialization_index: same=True
    DKRZ: np.int32(1)
    IPSL: np.int32(1)
  realization_index: same=True
    DKRZ: np.int32(10)
    IPSL: np.int32(10)
  forcing_index: same=True
    DKRZ: np.int32(2)
    IPSL: np.int32(2)
  physics_index: same=True
    DKRZ: np.int32(1)
    IPSL: np.int32(1)
  variant_label: same=True
    DKRZ: 'r10i1p1f2'
    IPSL: 'r10i1p1f2'
  startdate: same=True
    DKRZ: 's199511'
    IPSL: 's199511'
  parent_activi

In [9]:
if required_services <= datasets.keys():
    try:
        dkrz_tas, ipsl_tas = xr.align(dkrz["tas"], ipsl["tas"], join="exact")
    except ValueError as exc:
        print(f"Cannot compare tas values because coordinates differ: {exc}")
    else:
        dkrz_tas = dkrz_tas.load()
        ipsl_tas = ipsl_tas.load()
        difference = ipsl_tas - dkrz_tas
        absolute_difference = np.abs(difference)

        print("tas values (IPSL vs DKRZ)")
        print(f"  same dimensions: {dkrz_tas.dims == ipsl_tas.dims}")
        print(f"  same dtype: {dkrz_tas.dtype == ipsl_tas.dtype}")
        print(f"  exactly equal: {dkrz_tas.equals(ipsl_tas)}")
        print(
            "  all close (rtol=1e-05, atol=1e-08): "
            f"{np.allclose(dkrz_tas.values, ipsl_tas.values, rtol=1e-5, atol=1e-8, equal_nan=True)}"
        )
        print(
            "  same missing-value mask: "
            f"{np.array_equal(dkrz_tas.isnull().values, ipsl_tas.isnull().values)}"
        )
        print(
            "  maximum absolute difference: "
            f"{float(absolute_difference.max(skipna=True).item())}"
        )
        print(
            "  mean absolute difference: "
            f"{float(absolute_difference.mean(skipna=True).item())}"
        )


tas values (IPSL vs DKRZ)
  same dimensions: True
  same dtype: True
  exactly equal: True
  all close (rtol=1e-05, atol=1e-08): True
  same missing-value mask: True
  maximum absolute difference: 0.0
  mean absolute difference: 0.0


### Explain the dataset-level inequality

The complete datasets are not directly equal because the two services order dimensions differently on the latitude and longitude bounds variables. Compare their values again after transposing the DKRZ bounds to the IPSL dimension order.


In [10]:
if required_services <= datasets.keys():
    for variable_name in ("lat_bnds", "lon_bnds"):
        dkrz_bounds = dkrz[variable_name]
        ipsl_bounds = ipsl[variable_name]
        same_dimension_names = set(dkrz_bounds.dims) == set(ipsl_bounds.dims)

        print(f"{variable_name}")
        print(f"  DKRZ dimensions: {dkrz_bounds.dims}")
        print(f"  IPSL dimensions: {ipsl_bounds.dims}")
        print(f"  same attributes: {dkrz_bounds.attrs == ipsl_bounds.attrs}")
        if same_dimension_names:
            reordered_dkrz_bounds = dkrz_bounds.transpose(*ipsl_bounds.dims)
            print(
                "  same values after matching dimension order: "
                f"{reordered_dkrz_bounds.equals(ipsl_bounds)}"
            )


lat_bnds
  DKRZ dimensions: ('time', 'realization', 'lat', 'bnds')
  IPSL dimensions: ('realization', 'time', 'lat', 'bnds')
  same attributes: True
  same values after matching dimension order: True
lon_bnds
  DKRZ dimensions: ('time', 'realization', 'lon', 'bnds')
  IPSL dimensions: ('realization', 'time', 'lon', 'bnds')
  same attributes: True
  same values after matching dimension order: True


## Conclusion

The DKRZ and IPSL outputs are scientifically equivalent. The `tas` values are exactly equal, including their missing-value masks; the 24 monthly `360_day` time values are identical; the decadal `realization`, `reftime`, and `leadtime` coordinates are identical; and all global, data-variable, and coordinate attributes are identical.

The differences are limited to output packaging and dimension order: DKRZ returns two yearly batch files while IPSL returns one two-year file, and `lat_bnds`/`lon_bnds` use `(time, realization, lat/lon, bnds)` at DKRZ versus `(realization, time, lat/lon, bnds)` at IPSL. After transposing those bounds variables to the same dimension order, their values are exactly equal as well.
